# Gender Classification using Neural Network

**1. Importing the Dataset from Kaggle**

In [ ]:
import kagglehub

# Gender Classification Dataset by Ashish Jangra on Kaggle
path = kagglehub.dataset_download("ashishjangra27/gender-recognition-200k-images-celeba")

print("Path to dataset files:", path)

**2. Loading the Data**

In [ ]:
import tensorflow as tf

IMG_SIZE = (256, 256)
BATCH_SIZE = 32

# Importing the Data

# Training Data
train_data = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/input/datasets/ashishjangra27/gender-recognition-200k-images-celeba/Dataset/Train',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Validation Data
val_data = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/input/datasets/ashishjangra27/gender-recognition-200k-images-celeba/Dataset/Validation',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Testing Data
test_data = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/input/datasets/ashishjangra27/gender-recognition-200k-images-celeba/Dataset/Test',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# Normalising the Data
train_data = train_data.map(lambda x, y: (x/255, y))
val_data = val_data.map(lambda x, y: (x/255, y))
test_data = test_data.map(lambda x, y: (x/255, y))

In [ ]:
# Verifying the Data Size
train_size = int(len(train_data))
val_size = int(len(val_data))
test_size = int(len(test_data))
total_size = int(len(train_data)) + int(len(val_data)) + int(len(test_data))

print("Total Data Size: ", total_size)
print(f'Train Size: {train_size}\nValidation Size: {val_size}\nTest Size: {test_size}')

**3. Training the Model**

In [ ]:
# Creating the Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
model = Sequential()

# Adding Layers
model.add(Conv2D(64, (3,3), 1, activation='relu', input_shape=(256, 256, 3)))
model.add(MaxPooling2D())

model.add(Conv2D(32, (3,3), 1, activation='relu'))
model.add(MaxPooling2D())

model.add(Conv2D(16, (3,3), 1, activation='relu'))
model.add(MaxPooling2D())

model.add(Conv2D(16, (3,3), activation='relu'))
model.add(MaxPooling2D())

# Flatten Layer
model.add(Flatten())

# Dropout
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.5))

# Denser Layer
model.add(Dense(256, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.compile('adam', loss=tf.losses.BinaryCrossentropy(), metrics=['accuracy'])
model.summary()

In [ ]:
# Logging the Parameters
log_dir = 'logs'

# Tensorboard logs
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir)

# Training the Model
trained_model = model.fit(train_data, epochs=10, validation_data=val_data, callbacks=[tensorboard_callback])

**4. Evaluating Performance Metrics on Test Data**

In [ ]:
import numpy as np

y_true = []
y_pred = []

for images, labels in test_data:
    preds = model.predict(images)

    y_true.extend(labels.numpy())
    y_pred.extend(preds)

# Converting to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Converting probabilities to binary
y_pred_binary = (y_pred > 0.5).astype(int)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_true, y_pred_binary)
precision = precision_score(y_true, y_pred_binary)
recall = recall_score(y_true, y_pred_binary)
f1 = f1_score(y_true, y_pred_binary)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

In [ ]:
import matplotlib.pyplot as plt

metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
values = [accuracy, precision, recall, f1]

plt.figure()
plt.bar(metrics, values)

plt.title("Model Performance Metrics")
plt.ylabel("Score")
plt.ylim(0, 1)

for i, v in enumerate(values):
    plt.text(i, v + 0.02, f"{v:.2f}", ha='center')

plt.show()

**5. Predicting Random Images**

In [ ]:
import cv2
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Image Path
img_path = '/kaggle/input/datasets/ashishjangra27/gender-recognition-200k-images-celeba/Dataset/Test/Female/160004.jpg'  # change this

# Load image
img = cv2.imread(img_path)

# Convert BGR to RGB
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Resize
resize = tf.image.resize(img_rgb, (256, 256))

# Normalize
img_input = resize / 255.0

# Add batch dimension
img_input = np.expand_dims(img_input, axis=0)

# Predict
prediction = model.predict(img_input)[0][0]

# Convert to label
label = "Male" if prediction > 0.5 else "Female"

# Display image with label
plt.imshow(resize.numpy().astype(int))
plt.title(f"Prediction: {label}")
plt.axis('off')
plt.show()

**Saving the Model**

In [ ]:
from tensorflow.keras.models import load_model
model.save('/kaggle/working/gender_classifier_model.h5')

# To load for later use
model = load_model('/kaggle/working/gender_classifier_model.h5')